# Scroll pace report

Reads the local FastAPI service (or SQLite) to characterize sessions/dwell from the Android logger and the DYI URL catalog.

**Never** treat DYI inter-row gaps as dwell.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from urllib.request import urlopen, Request

import pandas as pd
import matplotlib.pyplot as plt

BASE = "http://127.0.0.1:8000"
TOKEN = ""  # set if API_TOKEN is configured


def api_get(path: str):
    req = Request(BASE.rstrip("/") + path)
    if TOKEN:
        req.add_header("Authorization", f"Bearer {TOKEN}")
    with urlopen(req) as resp:
        return json.loads(resp.read().decode("utf-8"))


user = api_get("/devices/users/me")
USER_ID = user["id"]
print("user", USER_ID, user.get("timezone"))

In [ ]:
sessions = pd.DataFrame(api_get(f"/sessions?user_id={USER_ID}"))
days = pd.DataFrame(api_get(f"/sessions/days?user_id={USER_ID}"))
impressions = pd.DataFrame(api_get(f"/impressions?user_id={USER_ID}"))

print("sessions", len(sessions), "impressions", len(impressions))
display(days)

In [ ]:
if len(impressions):
    dwell = impressions["dwell_ms"].dropna() / 1000.0
    skip_rate = float((dwell < 1.5).mean()) if len(dwell) else float("nan")
    print({
        "n_impressions": int(len(impressions)),
        "mean_dwell_s": float(dwell.mean()) if len(dwell) else None,
        "median_dwell_s": float(dwell.median()) if len(dwell) else None,
        "skip_rate_lt_1_5s": skip_rate,
        "with_url": int(impressions["external_url"].notna().sum()),
    })
    dwell.hist(bins=30)
    plt.xlabel("dwell (s)")
    plt.ylabel("count")
    plt.title("Impression dwell (logger only)")
    plt.show()
else:
    print("No impressions yet — run the Android logger + ingest first.")

## DYI catalog (optional local file)

If you imported via `scripts/import_dyi.py`, content records live in the DB. For a quick file-side check without the API:

In [ ]:
export_path = Path(r"C:\path\to\your-unzipped-instagram-export\ads_information\ads_and_topics\videos_watched.json")
if export_path.exists():
    rows = json.loads(export_path.read_text(encoding="utf-8"))
    urls = []
    for r in rows:
        for lv in r.get("label_values") or []:
            if lv.get("label") == "URL" and lv.get("href"):
                urls.append(lv["href"])
                break
    print("dyi watches", len(rows), "unique urls", len(set(urls)), "reel urls", sum("/reel/" in u for u in urls))
else:
    print("DYI file not found at", export_path)